# 07 — Day 7 Evaluation: WavLM-base-plus Layer Selection

**Standalone**, same pattern as `04_evaluate_trillsson.ipynb` and `permutation_test_hubert_layer12.ipynb`
— duplicated helper functions, no shared imports from `02_train_evaluate.ipynb`.

**What this tests:** all 13 WavLM-base-plus hidden layers, evaluated independently — plain 1536-dim
mean+std pooling per layer, the same 10-candidate set (`make_candidates`, no PCA), no session-fusion,
no augmentation. Isolated the same way as every prior layer sweep (Day 2's wav2vec2, Day 3's
HuBERT) so the embedding-family variable alone is being tested — nothing else changes.

**Inputs needed (Kaggle "Add Input"):**
1. The **`Fatigue_data`** dataset — for `labels.csv`.
2. **`07_features_wavlm.ipynb`**'s output — for `wavlm_base_plus_all_layers.parquet`.

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_SEED = 42
N_LAYERS = 13

LABELS_CSV = next(Path("/kaggle/input").rglob("labels.csv"))
WAVLM_LAYERS_PARQUET = next(Path("/kaggle/input").rglob("wavlm_base_plus_all_layers.parquet"))

print(f"LABELS_CSV           = {LABELS_CSV}")
print(f"WAVLM_LAYERS_PARQUET = {WAVLM_LAYERS_PARQUET}")

## 1. Load & Merge

In [ ]:
labels = pd.read_csv(LABELS_CSV)
wavlm_layers = pd.read_parquet(WAVLM_LAYERS_PARQUET)

df = labels.merge(wavlm_layers, on="filename", how="inner")
assert len(df) == len(labels) == 280, f"expected 280 rows after merge, got {len(df)}"

LABEL_MAP = {"non_fatigued": 0, "fatigued": 1}
df["y"] = df["label"].map(LABEL_MAP)
assert df["y"].isna().sum() == 0, "unmapped label value found"

y = df["y"].to_numpy()
groups = df["participant_id"].to_numpy()

print(f"{len(df)} rows, {len(np.unique(groups))} participants, {N_LAYERS} WavLM layers available")

## 2. Candidates & Nested CV (duplicated from `02_train_evaluate.ipynb`)

Same 10 candidates, same outer-LOSO -> inner-`GroupKFold(6)` -> per-fold inner-CV threshold
algorithm, same leak-free asserts used everywhere else in this project.

In [ ]:
CANDIDATE_CS = [0.001, 0.01, 0.1, 1.0, 10.0]


def build_pipeline(model_name: str, c: float) -> Pipeline:
    if model_name == "logreg":
        clf = LogisticRegression(C=c, max_iter=5000, class_weight="balanced")
    elif model_name == "linsvc":
        clf = LinearSVC(C=c, max_iter=10000, class_weight="balanced")
    else:
        raise ValueError(model_name)
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])


def make_candidates():
    return [
        (name, c, build_pipeline(name, c))
        for name in ["logreg", "linsvc"]
        for c in CANDIDATE_CS
    ]


def score_fn(fitted_pipeline: Pipeline, X_in: np.ndarray) -> np.ndarray:
    clf = fitted_pipeline.named_steps["clf"]
    if hasattr(clf, "predict_proba"):
        return fitted_pipeline.predict_proba(X_in)[:, 1]
    return fitted_pipeline.decision_function(X_in)


def best_threshold(scores: np.ndarray, y_true: np.ndarray) -> float:
    best_t, best_ba = 0.5, -1.0
    for t in np.unique(scores):
        ba = balanced_accuracy_score(y_true, (scores >= t).astype(int))
        if ba > best_ba:
            best_ba, best_t = ba, t
    return best_t

In [ ]:
def nested_loso_cv(X, y, groups, n_inner_splits=6):
    participants = np.unique(groups)
    oof_scores = np.zeros(len(y), dtype=float)
    oof_preds = np.zeros(len(y), dtype=int)
    fold_records = []
    n_outer_checked, n_inner_checked = 0, 0

    for test_participant in participants:
        outer_test_mask = groups == test_participant
        outer_train_mask = ~outer_test_mask
        assert not (set(groups[outer_train_mask]) & set(groups[outer_test_mask])), \
            f"outer leakage: {test_participant} present in both outer train and test"
        n_outer_checked += 1

        X_tr, y_tr, g_tr = X[outer_train_mask], y[outer_train_mask], groups[outer_train_mask]
        X_te = X[outer_test_mask]

        candidates = make_candidates()
        cand_aucs = {i: [] for i in range(len(candidates))}
        cand_scores = {i: [] for i in range(len(candidates))}
        cand_labels = {i: [] for i in range(len(candidates))}

        gkf = GroupKFold(n_splits=n_inner_splits)
        for inner_tr_idx, inner_val_idx in gkf.split(X_tr, y_tr, groups=g_tr):
            assert not (set(g_tr[inner_tr_idx]) & set(g_tr[inner_val_idx])), \
                f"inner leakage within outer fold {test_participant}"
            n_inner_checked += 1

            X_itr, y_itr = X_tr[inner_tr_idx], y_tr[inner_tr_idx]
            X_ival, y_ival = X_tr[inner_val_idx], y_tr[inner_val_idx]
            if len(np.unique(y_itr)) < 2 or len(np.unique(y_ival)) < 2:
                continue

            for i, (_, _, pipe) in enumerate(candidates):
                fitted = clone(pipe).fit(X_itr, y_itr)
                s = score_fn(fitted, X_ival)
                cand_aucs[i].append(roc_auc_score(y_ival, s))
                cand_scores[i].append(s)
                cand_labels[i].append(y_ival)

        mean_aucs = {i: np.mean(v) for i, v in cand_aucs.items() if v}
        best_i = max(mean_aucs, key=mean_aucs.get)
        best_name, best_c, best_template = candidates[best_i]

        pooled_inner_scores = np.concatenate(cand_scores[best_i])
        pooled_inner_labels = np.concatenate(cand_labels[best_i])
        threshold = best_threshold(pooled_inner_scores, pooled_inner_labels)

        final_pipe = clone(best_template).fit(X_tr, y_tr)
        test_scores = score_fn(final_pipe, X_te)
        oof_scores[outer_test_mask] = test_scores
        oof_preds[outer_test_mask] = (test_scores >= threshold).astype(int)

        fold_records.append({
            "test_participant": test_participant,
            "n_test": int(outer_test_mask.sum()),
            "chosen_model": best_name,
            "chosen_C": best_c,
            "inner_cv_auc": mean_aucs[best_i],
            "threshold": threshold,
        })

    print(f"  Verified zero participant overlap: {n_outer_checked} outer folds, "
          f"{n_inner_checked} inner GroupKFold(6) splits.")
    return oof_scores, oof_preds, pd.DataFrame(fold_records)


def bootstrap_auc_ci(participant_ids, y_true, scores, n_boot=1000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    participants = np.unique(participant_ids)
    idx_by_participant = {p: np.where(participant_ids == p)[0] for p in participants}
    boot_aucs = []
    for _ in range(n_boot):
        sampled = rng.choice(participants, size=len(participants), replace=True)
        idx = np.concatenate([idx_by_participant[p] for p in sampled])
        y_boot, s_boot = y_true[idx], scores[idx]
        if len(np.unique(y_boot)) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_boot, s_boot))
    lo, hi = np.percentile(boot_aucs, [2.5, 97.5])
    return lo, hi, np.array(boot_aucs)

## 3. Per-Layer Sweep

Same protocol repeated 13 times, once per hidden layer, plain 1536-dim mean+std pooling, no PCA, no
fusion, no augmentation — isolating the embedding-family variable alone.

In [ ]:
wavlm_layer_results = []
for L in range(N_LAYERS):
    layer_cols = [c for c in df.columns if c.startswith(f"layer{L}_emb_")]
    assert len(layer_cols) == 1536, f"expected 1536 layer-{L} columns, got {len(layer_cols)}"
    X_layer = df[layer_cols].to_numpy()

    print(f"=== WavLM layer {L} ===")
    oof_L, preds_L, fold_info_L = nested_loso_cv(X_layer, y, groups)
    auc_L = roc_auc_score(y, oof_L)
    ci_lo_L, ci_hi_L, _ = bootstrap_auc_ci(groups, y, oof_L)

    wavlm_layer_results.append({"layer": L, "auc": auc_L, "ci_lo": ci_lo_L, "ci_hi": ci_hi_L})
    print(f"  AUC={auc_L:.3f}  CI=[{ci_lo_L:.3f}, {ci_hi_L:.3f}]\n")

wavlm_layer_results_df = pd.DataFrame(wavlm_layer_results)

## 4. Compare Against wav2vec2 (Day 2) and HuBERT (Day 3)

Reference curves hardcoded from `PROJECT_PLAN.md`'s already-logged Day 2/Day 3 results, since this
notebook is standalone and doesn't load `02_train_evaluate.ipynb`'s state. All three models share
the same architecture shape (13 layers, 768 hidden dim), so layer indices line up directly.

In [ ]:
# Hardcoded from PROJECT_PLAN.md's Day 2 / Day 3 results tables.
WAV2VEC2_DAY2_AUC = [0.534, 0.560, 0.579, 0.574, 0.549, 0.569, 0.538,
                     0.581, 0.577, 0.575, 0.535, 0.560, 0.542]
HUBERT_DAY3_AUC = [0.537, 0.572, 0.517, 0.571, 0.585, 0.537, 0.558,
                   0.581, 0.545, 0.604, 0.565, 0.581, 0.611]
assert len(WAV2VEC2_DAY2_AUC) == len(HUBERT_DAY3_AUC) == N_LAYERS

WAV2VEC2_COLOR = "#2a78d6"   # slot 1 blue -- Day 2
HUBERT_COLOR = "#eb6834"     # slot 2 orange -- Day 3
WAVLM_COLOR = "#1baf7a"      # slot 3 aqua -- Day 7
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(N_LAYERS), WAV2VEC2_DAY2_AUC, color=WAV2VEC2_COLOR, marker="o",
        markersize=5, linewidth=1.5, label="wav2vec2-base (Day 2)")
ax.plot(range(N_LAYERS), HUBERT_DAY3_AUC, color=HUBERT_COLOR, marker="o",
        markersize=5, linewidth=1.5, label="HuBERT-base (Day 3)")
ax.plot(wavlm_layer_results_df["layer"], wavlm_layer_results_df["auc"], color=WAVLM_COLOR,
        marker="o", markersize=6, linewidth=2, label="WavLM-base-plus (Day 7)")
ax.axhline(0.5, color=MUTED, linewidth=1, linestyle="--")
ax.set_xticks(range(N_LAYERS))
ax.set_xlabel("Hidden layer (0 = embedding output, 12 = final transformer layer)", color=INK)
ax.set_ylabel("Pooled ROC-AUC", color=INK)
ax.tick_params(colors=MUTED)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(frameon=False, labelcolor=INK, loc="upper left")
plt.tight_layout()
plt.show()

best_wavlm = wavlm_layer_results_df.loc[wavlm_layer_results_df["auc"].idxmax()]
print(f"Best WavLM layer:    {int(best_wavlm['layer'])}  AUC={best_wavlm['auc']:.3f}  "
      f"CI=[{best_wavlm['ci_lo']:.3f}, {best_wavlm['ci_hi']:.3f}]")
print(f"Best wav2vec2 layer (Day 2): layer {int(np.argmax(WAV2VEC2_DAY2_AUC))}  "
      f"AUC={max(WAV2VEC2_DAY2_AUC):.3f}")
print(f"Best HuBERT layer (Day 3):   layer {int(np.argmax(HUBERT_DAY3_AUC))}  "
      f"AUC={max(HUBERT_DAY3_AUC):.3f}  (the confirmed Day 3 winner, p=0.01)")
print(f"\nDay 6 final pipeline for reference (HuBERT L12 + augmentation + session-fusion): "
      f"AUC=0.651, CI=[0.584, 0.720], permutation p=0.0020")

## How to read Day 7's results

- **WavLM's best layer clearly beats HuBERT's best layer (0.611)**, with a non-marginal CI (not a
  hair's-width clearance the way Day 1/2's early results were): WavLM becomes a genuine alternative
  base embedding worth carrying into Day 8 — combine WavLM's best layer with speed augmentation +
  session-fusion (Day 5/6's recipe) as a parallel candidate alongside the existing HuBERT pipeline,
  and/or use it as a second embedding in Day 8's fusion/ensembling experiments.
- **WavLM is close to HuBERT's 0.611 with substantially overlapping CIs**: no clear single-embedding
  win, but WavLM likely still carries *some* complementary signal (different pretraining objective) —
  worth keeping as an ensembling candidate for Day 8 rather than the sole new base embedding, since
  Day 6 already confirmed HuBERT L12 + augmentation + session-fusion at 0.651.
- **WavLM is clearly below HuBERT across most/all layers**: report as an honest negative result (the
  masked-denoising objective's paralinguistic sensitivity, documented on other benchmarks, didn't
  transfer to this specific fatigue task at this sample size) and proceed to Day 8 using HuBERT
  alone as the base embedding, dropping WavLM from further consideration rather than forcing it into
  an ensemble that likely wouldn't help.

**No permutation test here** — per this project's discipline since Day 1, that's reserved for a
single finalized candidate (Day 10's buffer), not run per-layer or per-variant during exploration.

Update `PROJECT_PLAN.md`'s Day 7+ roadmap section with today's numbers (best WavLM layer, and the
three-way comparison) before moving to Day 8.